# Air Quality ETL Pipeline

This notebook explains the end-to-end ETL pipeline for air quality data from Visual Crossing API.

## Architecture

```
Visual Crossing API
       ↓
Extract (historical / forecast / today_hourly)
       ↓
Raw CSV (data/raw/{historical, forecast, today_hourly}/)
       ↓
Clean & Validate (DataFrameCleaner, DataValidator, DataAuditor)
       ↓
Processed CSV (data/processed/daily_air_quality_combined.csv)
       ↓
Transform to Star Schema (dim_date, dim_city, fact_air_quality, fact_air_quality_today)
       ↓
Load to CSV (data/star_schema/) + PostgreSQL
```

## Key Components

- **AirQualityExtractor**: Fetches air quality data from Visual Crossing API with retry-on-429
- **CityExtractor**: Loads dim_city from CSV
- **DataFrameCleaner**: Normalizes empty strings, removes duplicates, fills nulls
- **DataValidator**: Range-checks air quality metrics (PM2.5, PM10, NO2, O3, CO, SO2, AQI)
- **DataAuditor**: Logs comprehensive DataFrame audit (nulls, duplicates, stats)
- **CsvLoader**: Saves raw, processed, and star-schema CSVs
- **PostgresLoader**: Saves star schema to PostgreSQL with constraints

## Data Flow

1. Settings are validated and directories created
2. Cities are loaded from dim_city.csv
3. For each city: historical (last 5 days), forecast (next 5 days), and today-hourly data are extracted
4. Raw data is saved to `data/raw/{type}/{city}_{type}.csv`
5. All data is cleaned via DataFrameCleaner
6. Daily data (historical + forecast) is combined and deduplicated
7. dim_date is generated for the current and next year
8. Fact tables are built and validated
9. Star schema is saved to CSV and PostgreSQL